<a href="https://colab.research.google.com/github/TommyS725/FTEC5660/blob/main/homeworks/hw2/homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2

# Set up

## Installing packages

In [13]:
!pip install requests PyPDF2 gdown
!pip install 'markitdown[pdf]'
!pip install langchain_mcp_adapters langchain_google_genai langchain-openai

## Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `VERTEX_API_KEY`.


1.   Look for the key icon on the left panel of your colab.
2.   Under `Name`, create `VERTEX_API_KEY`.
3. Copy your key to `Value`.

If you cannot use VERTEX_API_KEY, you can use deepseek models via `DEEPSEEK_API_KEY`. It does not affect your score.



In [14]:
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
# DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# Download sample CVs

## Downloading sample_cv.pdf
The codes below download the sample CV


In [19]:
import os
import gdown

folder_id = "1adYKq7gSSczFP3iikfA8Er-HSZP6VM7D"
folder_url = f"https://drive.google.com/drive/folders/{folder_id}"

output_dir = "downloaded_cvs"
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=folder_url,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

Retrieving folder contents


Processing file 1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp CV_1.pdf
Processing file 16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs CV_2.pdf
Processing file 15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr CV_3.pdf
Processing file 1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk CV_4.pdf
Processing file 1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C CV_5.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NR1RUKx4GyM7QOBxKXkfh4e8jUkxFCsp
To: /content/downloaded_cvs/CV_1.pdf
100%|██████████| 147k/147k [00:00<00:00, 6.36MB/s]
Downloading...
From: https://drive.google.com/uc?id=16lrd-uO8AAnCnv7UG9Rs_Nk6SUu0Iwbs
To: /content/downloaded_cvs/CV_2.pdf
100%|██████████| 75.1k/75.1k [00:00<00:00, 2.74MB/s]
Downloading...
From: https://drive.google.com/uc?id=15hVEuBan_EKhEty2aZBd6rcpDpP4o7Vr
To: /content/downloaded_cvs/CV_3.pdf
100%|██████████| 72.0k/72.0k [00:00<00:00, 2.95MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Y2w_mAUEhg4vZBdvvR-0n3Jf2mKuGDRk
To: /content/downloaded_cvs/CV_4.pdf
100%|██████████| 73.3k/73.3k [00:00<00:00, 3.40MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PLwkva-pdua6ZVvmLg9mxHeljq9D8C_C
To: /content/downloaded_cvs/CV_5.pdf
100%|██████████| 97.9k/97.9k [00:00<00:00, 3.48MB/s]
Download complete

['downloaded_cvs/CV_1.pdf',
 'downloaded_cvs/CV_2.pdf',
 'downloaded_cvs/CV_3.pdf',
 'downloaded_cvs/CV_4.pdf',
 'downloaded_cvs/CV_5.pdf']

In [20]:
# =====================================================
#  Load and display all CV PDFs in order
# =====================================================
import os
from markitdown import MarkItDown

cv_dir = "downloaded_cvs"

# Initialize MarkItDown
md = MarkItDown(enable_plugins=False)

# Collect and sort PDFs numerically
pdf_files = sorted(
    [f for f in os.listdir(cv_dir) if f.lower().endswith(".pdf")],
    key=lambda x: int("".join(filter(str.isdigit, x)))  # CV_1.pdf → 1
)

all_cvs = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(cv_dir, pdf_name)
    result = md.convert(pdf_path)

    all_cvs.append({
        "file": pdf_name,
        "text": result.text_content
    })

    print("=" * 80)
    print(f"📄 {pdf_name}")
    print("=" * 80)
    print(result.text_content)
    print("\n\n")


📄 CV_1.pdf
|     |     |     |     | John         |           | Smith        |                   |     |     |
| --- | --- | --- | --- | ------------ | --------- | ------------ | ----------------- | --- | --- |
|     |     |     |     | Marketing    |           | Professional |                   |     |     |
|     |     |     |     | + Singapore, | Singapore |              | (cid:209) Kowloon |     |     |
Experience
|                |                  |     |          |                     |              |            |     | 2020 – | Present |
| -------------- | ---------------- | --- | -------- | ------------------- | ------------ | ---------- | --- | ------ | ------- |
| Engineer,      | ByteDance        |     |          |                     |              |            |     |        |         |
| • Worked       | in a fast-paced, |     | global   | technology          | environment. |            |     |        |         |
| • Collaborated | across           |     | teams to | sup

# Connect to our MCP server

Documentation about MCP: https://modelcontextprotocol.io/docs/getting-started/intro.

Using MCP servers in Langchain https://docs.langchain.com/oss/python/langchain/mcp.

## Check which tools that the MCP server provide

In [15]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
for tool in mcp_tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)
    print("\n\n------------------------------------------------------\n\n")

search_facebook_users
Search for Facebook users by display name (supports partial and fuzzy matching).

Args:
    q: Search query string (case-insensitive, matches any part of display name)
       Examples: "John", "john smith", "Smith"
    limit: Maximum number of results to return (default: 20, max: 20)
    fuzzy: Enable fuzzy matching if exact search returns no results (default: True)

Returns:
    List of user dictionaries, each containing:
    - id (int): Unique Facebook user ID for use with get_facebook_profile()
    - display_name (str): User's Facebook display name (may differ from legal name)
    - city (str): Current city of residence
    - country (str): Country of residence
    - match_type (str): "exact" or "fuzzy" (indicates search method used)
    
    Returns empty list [] if no matches found.

Example:
    search_facebook_users("Alex Chan", limit=5)
    → [{"id": 123, "display_name": "Alex Chan", "city": "Hong Kong", "country": "Hong Kong", "match_type": "exact"}]
    

## A simple agent using tools from the MCP server


In [18]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# ---------------------------
# 1. Define a local tool
# ---------------------------
@tool
def say_hello(name: str) -> str:
    """Say hello to a person by name."""
    return f"Hello, {name}! 👋"

# ---------------------------
# 2. Load MCP tools + merge
# ---------------------------
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
tools = mcp_tools + [say_hello]

# ---------------------------
# 3. Initialize Gemini (tool-enabled) or deepseek
# ---------------------------
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    temperature=0,
    vertexai = True,
)

# from langchain_openai import ChatOpenAI
# DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
# llm = ChatOpenAI(
#     model="deepseek-chat",          # or "deepseek-reasoner"
#     api_key=DEEPSEEK_API_KEY,
#     base_url="https://api.deepseek.com/v1",
#     temperature=0,
# )

llm_with_tools = llm.bind_tools(tools)

# ---------------------------
# 4. Single-step invocation
# ---------------------------
query = "Say hello to Bao using tool, then search for someone named Alice on Facebook."

response = llm_with_tools.invoke([
    HumanMessage(content=query)
])

print(response)

content='' additional_kwargs={'function_call': {'name': 'search_facebook_users', 'arguments': '{"q": "Alice"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c7fe9-76eb-7ad3-8307-982bccf750ff-0' tool_calls=[{'name': 'say_hello', 'args': {'name': 'Bao'}, 'id': '0ea2bfe5-edf1-4e9f-bec4-551182cdb5ef', 'type': 'tool_call'}, {'name': 'search_facebook_users', 'args': {'q': 'Alice'}, 'id': '192170d0-1e1a-45be-b1f3-f8d17a96a0d3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 2568, 'output_tokens': 13, 'total_tokens': 2581, 'input_token_details': {'cache_read': 0}}


In [ ]:
# This block provides you some tests to get faminilar with our MCP server

# # Test 1: Search Facebook users (exact match)
# await tools[0].ainvoke({'q': "Alex Chan", 'limit': 5})

# # Test 2: Search Facebook users (fuzzy match with typo)
# await tools[0].ainvoke({'q': "Alx Chn", 'limit': 5, 'fuzzy': True})

# # Test 3: Get Facebook profile
# await tools[1].ainvoke({'user_id': 123})

# # Test 4: Get Facebook mutual friends
# await tools[2].ainvoke({'user_id_1': 123, 'user_id_2': 456})

# # Test 5: Search LinkedIn people (exact match)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5})

# # Test 6: Search LinkedIn people (fuzzy match with typo)
# await tools[3].ainvoke({'q': "Python", 'location': "Hong Kong", 'limit': 5, 'fuzzy': True})

# # Test 7: Get LinkedIn profile
# await tools[4].ainvoke({'person_id': 456})

# Test 8: Get LinkedIn interactions
await tools[5].ainvoke({'person_id': 456})

[{'type': 'text',
  'text': '{"profile_id":456,"post_count":4,"total_likes":5,"liked_by":[4390,3622,7500,4269,8464],"engagement_score":1.25}',
  'id': 'lc_cf7cf5a1-dc1c-470e-b927-19d225e687f5'}]

In [212]:
from pydantic import BaseModel, Field,confloat
from typing import List, Literal
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from itertools import batched
from langchain_core.messages import ToolMessage

import glob, os

In [153]:
import json
cv1_info = {
  "name": "John Smith",
  "locations": ["Singapore"],
  "education": [
    {
      "school": "McGill University",
      "degree": "Bachelor of Science (BSc) in Marketing",
      "graduation_year": 2009
    }
  ],
  "experience": [
    {
      "company": "ByteDance",
      "title": "Engineer",
      "start_year": 2020,
      "end_year": "Present",
      "location": "Singapore"
    }
  ],
  "skills": [
    "Content Creation",
    "SEO",
    "Social Media"
  ],
  "current_title": "Marketing Professional"
}
text = json.dumps(cv1_info)
text

'{"name": "John Smith", "locations": ["Singapore"], "education": [{"school": "McGill University", "degree": "Bachelor of Science (BSc) in Marketing", "graduation_year": 2009}], "experience": [{"company": "ByteDance", "title": "Engineer", "start_year": 2020, "end_year": "Present", "location": "Singapore"}], "skills": ["Content Creation", "SEO", "Social Media"], "current_title": "Marketing Professional"}'

In [87]:
async def call(tools,tool_name,args):
  print(tool_name,args)
  for tool in tools:
    if tool.name == tool_name:
      try:
        if hasattr(tool, "ainvoke") :
          return await tool.ainvoke(args)
        elif hasattr(tool, "invoke"):
          return tool.invoke(args)
        else:
          continue
      except Exception as e:
        return f"Error invoking {tool.name}: {str(e)}"
  return f"Error: Tool {tool_name} not found."

In [30]:
for tool in mcp_tools:
  print(tool.name)

search_facebook_users
get_facebook_profile
get_facebook_mutual_friends
search_linkedin_people
get_linkedin_profile
get_linkedin_interactions


In [53]:
await call(mcp_tools,"search_facebook_users",{'q': "Alex Chan", 'limit': 5})

[{'type': 'text',
  'text': '[{"id":3,"display_name":"Alex Chan","city":"Hyderabad","country":"India","match_type":"exact"},{"id":41,"display_name":"Alex Chan","city":"Shanghai","country":"China","match_type":"exact"},{"id":79,"display_name":"Alex Chan","city":"London","country":"UK","match_type":"exact"},{"id":106,"display_name":"Alex Chan","city":"Ho Chi Minh City","country":"Vietnam","match_type":"exact"},{"id":117,"display_name":"Alex Chan","city":"Melbourne","country":"Australia","match_type":"exact"}]',
  'id': 'lc_2cc4da16-aedb-4921-a409-bd6fdd584b98'}]

In [162]:
async def facebook_agent_loop(query: str, max_turns: int = 30, verbose=False, _llm = llm, _tools=mcp_tools):
  system_prompt =  """
You are the **Facebook Verification Agent** in a CV verification system using the SocialGraph MCP server.

Tools available:
- `search_facebook_users(q, limit, fuzzy)` to search for candidates by name and location.
- `get_facebook_profile(userid)` to retrieve a detailed Facebook profile.
- Optionally `get_facebook_mutual_friends(userid1, userid2)` for extra identity consistency checks.

Input:
- Structured candidate info from the Main Agent, including:
  - name
  - locations (current and past)
  - education (schools, degrees, years)
  - experience (companies, titles, years, locations)
  - skills and interests (if any)

Your tasks:
1. Identify the most likely matching Facebook profile using name plus location (and other hints if useful).
2. Retrieve the profile details.
3. Compare CV claims against Facebook information:
   - name consistency
   - city / country
   - current job title and company
   - education (school, degree, years when visible)
   - interests / bio if relevant

STEPS:
1. Call `search_facebook_users(limit=20)`
2. Call `get_facebook_profile()` on the ALL prfile in the SAME tool call for comparison
3. Decide most similar existing profile
4. Generate the rpeort according to [Output format]

Guidelines:
  - In teh first step, search_facebook_users(limit=20), ALWAYS call with limit=20
  - When answering query, you MUST always call tools to fetch latest information, instead of using internal knowledge.
	-	If no reasonable match exists, set  match_confidence  to  none ,  overall_score  to 0, and explain why.
	-	Do NOT include raw private data (e.g., full post histories); only summarize relevant signals.

Output format:
Match Confidence: high|medium|low|none
Overall Score: 0-100
Matched Fields
	- 	field_name: CV=”…” matches FB=”…” Comment: why it matches
Mismatched Fields
	-	field_name: CV=”…” vs FB=”…” minor/major Comment: explanation
Uncertain Fields
	-	field_name: Reason why uncertain
Profile Summary:
2-4 sentences explaining overall alignment
Cross-Platform Notes (for LinkedIn & Report Agent)
	- Must-Check Items for LinkedIn:
	-	Potential Red Flags:
	-	Key Fields That Should Agree Across Platforms:
  - Missing in Facebook and good to have in LinkedIn

....
"""
  llm = _llm
  llm_tools = llm.bind_tools(_tools)
  history = [
        ("system", system_prompt),
        ("human", f"Query: {query}")
    ]
  tool_call_cnt = 0
  for turn in range(max_turns):
        # LLM generates next step (could be tool call or final answer)
        response = llm_tools.invoke(history)
        # If LLM provides a final (non-tool) answer, return it
        if not response.tool_calls:
            # handle sometime llm answer without trying call tool, force try again
            if tool_call_cnt == 0:
              print(response)
              continue
            print(history)
            return response.content
        history.append(response)

        # Handle tool calls
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"].lower()
            args = tool_call["args"]
            if verbose:
                print(f"\n[Turn {turn+1}] Calling Tool: {tool_name}")

            observation = await call(_tools,tool_name,args)

            # Feed the tool's output (or error) back to the LLM for the next step
            history.append(ToolMessage(
                tool_call_id=tool_call["id"],
                content=str(observation)
            ))
            tool_call_cnt +=1
            if verbose:
                print(f"[Observation]: {observation}")

  return "Loop reached max turns without a final answer."


In [127]:
text

'{"name": "John Smith", "locations": ["hong kong"], "education": [{"school": "McGill University", "degree": "Bachelor of Science (BSc) in Marketing", "graduation_year": 2009}], "experience": [{"company": "ByteDance", "title": "Engineer", "start_year": 2020, "end_year": "Present", "location": "Singapore"}], "skills": ["Content Creation", "SEO", "Social Media"], "current_title": "Marketing Professional"}'

In [163]:
fb = await facebook_agent_loop(text,verbose=True)
print(fb)


[Turn 1] Calling Tool: search_facebook_users
search_facebook_users {'limit': 20, 'q': 'John Smith'}
[Observation]: [{'type': 'text', 'text': '[{"id":8,"display_name":"John Smith","city":"Kowloon","country":"Hong Kong","match_type":"exact"},{"id":51,"display_name":"John Smith","city":"Central","country":"Hong Kong","match_type":"exact"},{"id":213,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},{"id":254,"display_name":"John Smith","city":"Causeway Bay","country":"Hong Kong","match_type":"exact"},{"id":322,"display_name":"John Smith","city":"Munich","country":"Germany","match_type":"exact"},{"id":323,"display_name":"John Smith","city":"Ho Chi Minh City","country":"Vietnam","match_type":"exact"},{"id":335,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},{"id":377,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},{"id":407,"display_name":"John Smith","city":"Shangh

In [169]:
async def linkedin_agent_loop(query: str, max_turns: int = 20, verbose=False, _llm = llm, _tools=mcp_tools):
  system_prompt = """
  You are the **LinkedIn Verification Agent** in a CV verification system using the SocialGraph MCP server.

Tools available:
- `search_linkedin_people(q, location, industry, limit, fuzzy)` to find candidate profiles.
- `get_linkedin_profile(personid)` for detailed professional profile data.
- `get_linkedin_interactions(personid)` to measure network activity and engagement.

Input:
- Structured candidate info from the Main Agent:
  - name
  - locations
  - education (school, degree, years)
  - experience (company, title, years, location)
  - skills

Your tasks:
1. Search LinkedIn for the candidate, using name plus location (and industry if appropriate).
2. Select the best-matching profile and retrieve details.
3. Compare CV claims against LinkedIn:
   - employment history: companies, titles, start/end years, current role
   - education: schools, degrees, years
   - skills and seniority vs CV claims
   - status if relevant (e.g., employed vs unemployed)
4. Optionally use interactions as a weak signal for how established the profile is.
5. Return a structured result:

Guidelines:
  - When answering query, you MUST always call tools to fetch latest information, instead of using internal knowledge.
	-	Be strict with serious mismatches (e.g., different company names, impossible timelines).
	-	If multiple profiles are plausible, pick the best one but record the uncertainty.
	-	If nothing matches, set  match_confidence  to  none ,  overall_score  to 0, and explain briefly.

  Output Format:
  Match Confidence: high|medium|low|none
Overall Score: 0-100
Matched Fields
	-	field_name: CV=”…” matches LinkedIn=”…” Comment: why it matches
Mismatched Fields
	-	field_name: CV=”…” vs LinkedIn=”…” minor/major Comment: explanation
Uncertain Fields
	-	field_name: Reason why uncertain
Profile Summary:
2-4 sentences explaining overall alignment
Cross-Platform Notes (for Facebook & Report Agent)
	-	Response to Expected Facebook Concerns:
	-	Key Agreements with Facebook (if known):
	-	Key Disagreements to Highlight:
	-	Fields Critical for Final Risk Assessment:

  """

  llm = _llm
  llm_tools = llm.bind_tools(_tools)
  history = [
        ("system", system_prompt),
        ("human", f"Query: {query}")
    ]
  for turn in range(max_turns):
        # LLM generates next step (could be tool call or final answer)
        response = llm_tools.invoke(history)
        history.append(response)
        # If LLM provides a final (non-tool) answer, return it
        if not response.tool_calls:
            return response.content

        # Handle tool calls
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"].lower()
            args = tool_call["args"]
            if verbose:
                print(f"\n[Turn {turn+1}] Calling Tool: {tool_name}")

            observation = await call(_tools,tool_name,args)

            # Feed the tool's output (or error) back to the LLM for the next step
            history.append(ToolMessage(
                tool_call_id=tool_call["id"],
                content=str(observation)
            ))
            if verbose:
                print(f"[Observation]: {observation}")

  return "Loop reached max turns without a final answer."


In [171]:
li = await linkedin_agent_loop(text,verbose=True)
print(li)


[Turn 1] Calling Tool: search_linkedin_people
search_linkedin_people {'location': 'Singapore', 'q': 'John Smith'}
[Observation]: [{'type': 'text', 'text': '[{"id":9,"name":"John Smith","headline":"Marketing Professional","industry":"Marketing","location":"Singapore, Singapore","years_experience":7,"match_type":"exact"},{"id":31,"name":"John Smith","headline":"Logistics Professional","industry":"Logistics","location":"Singapore, Singapore","years_experience":18,"match_type":"exact"},{"id":203,"name":"John Smith","headline":"Legal Professional","industry":"Legal","location":"Singapore, Singapore","years_experience":15,"match_type":"exact"},{"id":213,"name":"John Smith","headline":"AI Professional","industry":"AI","location":"Singapore, Singapore","years_experience":8,"match_type":"exact"},{"id":335,"name":"John Smith","headline":"Logistics Professional","industry":"Logistics","location":"Singapore, Singapore","years_experience":2,"match_type":"exact"},{"id":377,"name":"John Smith","head

In [213]:

RiskLevel = Literal["low", "medium", "high"]
Severity = Literal["minor", "major"]


class Discrepancy(BaseModel):
    field: str = Field(..., description="The profile field where the discrepancy was found (e.g., experience, education).")
    description: str = Field(..., description="What doesn't match and why it was flagged.")
    severity: Severity


class DetailedReport(BaseModel):
    facebook_section: str = Field(..., description="Findings extracted from Facebook.")
    linkedin_section: str = Field(..., description="Findings extracted from LinkedIn.")
    discrepancies: List[Discrepancy] = Field(default_factory=list)
    recommendations: List[str] = Field(default_factory=list)


class KYCReport(BaseModel):
    final_score: confloat(ge=0, le=100) = Field(..., description="Overall risk score from 0 to 100.")
    risk_level: RiskLevel
    summary: str = Field(..., description="2–4 sentence high-level summary.")
    detailed_report: DetailedReport
    @property
    def text(self) -> None:
        return self.model_dump_json(indent=2, exclude_none=True)

def report_agent(cv,fb,li,_llm=llm):
  system_prompt = """You are the Report Agent for a CV verification system.
Input:
	•	 cv_structured : structured information extracted from the CV by the Main Agent.
	•	 facebook_verification : structured result from the Facebook Agent.
	•	 linkedin_verification : structured result from the LinkedIn Agent.
Your tasks:
	1.	Synthesize a clear verification report for a human reviewer.
	2.	Highlight:
	•	Overall alignment between CV and social profiles.
	•	Key matches that support the candidate’s claims.
	•	Key discrepancies, with severity assessment.
	•	Areas of uncertainty (missing or ambiguous data).
	3.	Produce a final numeric score between 0 and 100 that represents how trustworthy the CV is, based on:
	•	Facebook and LinkedIn  overall_score  values.
	•	Match confidence and severity of mismatches.
	•	Presence of major red flags (e.g., different company names, impossible timelines).
"""
  llm_struct = _llm.with_structured_output(KYCReport)
  history = [
        ("system", system_prompt),
        ("human", f"[cv_structured]\n {cv}\nfacebook_verification: {fb}\n[linkedin_verification]:\n{li}"),

    ]
  res = llm_struct.invoke(history)
  # format score to 0-1 scale
  res.final_score/=100
  return res


In [214]:
rep = report_agent(text,fb,li)
print(rep.text)

In [226]:
def extract_cv_agent(text:str,_llm=llm):
  system_prompt = """You are the **Main CV Extractor** for a CV verification system.
Your ONLY job is to read CV content and extract structured candidate information.

**DO NOT CALL ANY TOOLS. DO NOT VERIFY ANYTHING. EXTRACTION ONLY.**

**REQUIRED OUTPUT FORMAT** (use this exact JSON structure):

{
  "name": string|null,
  "locations": string[],
  "current_title": string|null,
  "education": [{"school": string|null, "degree": string|null, "graduation_year": number|null}],
  "experience": [{"company": string|null, "title": string|null, "start_year": number|null, "end_year": "Present"|number|null, "location": string|null}],
  "skills": string[],
  "extra context":string|null
}


Guidelines:
	-	Extract ALL relevant fields from the CV. Never fabricate data.
	-	If info is missing/unclear, use  null  or empty lists  [] .
	- Parse tables, headers, and bullet points carefully.
	-	Normalize years (e.g., “2020 – Present” → start_year: 2020, end_year: “Present”).
	-	Locations: list ALL mentioned (current + past).
	-	Be precise with company names and exact job titles.
	-	Skills: extract from dedicated skills section only.
Output ONLY the JSON above. No explanations or additional text."""

  history = [
    ('system', system_prompt),
    ('human',text)
  ]
  return  _llm.invoke(history).content



In [229]:
r = extract_cv_agent(all_cvs[0]["text"])
print(r)

```json
{
  "name": "John Smith",
  "locations": [
    "Singapore",
    "Kowloon"
  ],
  "current_title": "Engineer",
  "education": [
    {
      "school": "McGill University",
      "degree": "Bachelor of Science (BSc) in Marketing",
      "graduation_year": 2009
    }
  ],
  "experience": [
    {
      "company": "ByteDance",
      "title": "Engineer",
      "start_year": 2020,
      "end_year": "Present",
      "location": null
    }
  ],
  "skills": [
    "Content Creation",
    "SEO",
    "Social Media"
  ],
  "extra context": null
}
```


In [240]:
async def kyc_agent(cv,_llm=llm,verbose=False):
  verbose and print(f"[Extract Info from {cv["file"]}]")
  text = extract_cv_agent(cv['text'])
  verbose and print(text)
  verbose and print("[Verify from Facebook]")
  fb = await facebook_agent_loop(text,_llm=_llm,verbose=verbose)
  verbose and print(fb)
  verbose and print("[Verify from Linkedin]")
  li = await linkedin_agent_loop(text,_llm=_llm,verbose=verbose)
  verbose and print(li)
  verbose and print("[Generate Report]")
  report = report_agent(text,fb,li,_llm=_llm)
  verbose and print(report.text)
  print(f"[KYC flow for {cv["file"]} done]")
  return report



In [239]:
await kyc_agent(all_cvs[1],verbose=True)

[Extract Info from CV_2.pdf]
```json
{
  "name": "Minh Pham",
  "locations": [
    "Beijing, China",
    "Hong Kong"
  ],
  "current_title": "Manager",
  "education": [
    {
      "school": "The University of Hong Kong",
      "degree": "BSc in Design",
      "graduation_year": 2011
    }
  ],
  "experience": [
    {
      "company": "BCG",
      "title": "Manager",
      "start_year": 2022,
      "end_year": "Present",
      "location": null
    },
    {
      "company": "Tencent",
      "title": "Analyst",
      "start_year": 2013,
      "end_year": 2017,
      "location": null
    }
  ],
  "skills": [
    "UI/UX Design",
    "Prototyping",
    "Graphic Design"
  ],
  "extra context": null
}
```
[Verify from Facebook]

[Turn 1] Calling Tool: search_facebook_users
search_facebook_users {'limit': 20, 'q': 'Minh Pham'}
[Observation]: [{'type': 'text', 'text': '[{"id":62,"display_name":"Minh Pham","city":"Austin","country":"USA","match_type":"exact"},{"id":70,"display_name":"Minh Pham",

KYCReport(final_score=0.35, risk_level='medium', summary="The CV of Minh Pham presents several inconsistencies when compared to their Facebook and LinkedIn profiles. While the name and some skills align, significant discrepancies exist in current employment, past experience, and education. The Facebook profile suggests a current role at Manulife, while LinkedIn indicates Deloitte, contrasting with the CV's claim of BCG. Education details also vary across platforms, raising concerns about the accuracy and reliability of the CV.", detailed_report=DetailedReport(facebook_section="The Facebook profile of Minh Pham from Beijing shows some alignment with the CV, including name and location. Both profiles have education in Hong Kong. However, the current company (Manulife) and job title (Engineer) differ from the CV (BCG, Manager). There's no direct match for Tencent in the profile, but the friends list includes a Manager at Tencent, which aligns with the candidate's experience.", linkedin_se

In [235]:
for cv in all_cvs:
  print("="*50)
  report = await kyc_agent()
  print(report.text)
  print("="*50)


[Extract Info from CV_1.pdf]
[Verify from Facebook]

[Turn 1] Calling Tool: search_facebook_users
search_facebook_users {'limit': 20, 'q': 'John Smith'}
[Observation]: [{'type': 'text', 'text': '[{"id":8,"display_name":"John Smith","city":"Kowloon","country":"Hong Kong","match_type":"exact"},{"id":51,"display_name":"John Smith","city":"Central","country":"Hong Kong","match_type":"exact"},{"id":213,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},{"id":254,"display_name":"John Smith","city":"Causeway Bay","country":"Hong Kong","match_type":"exact"},{"id":322,"display_name":"John Smith","city":"Munich","country":"Germany","match_type":"exact"},{"id":323,"display_name":"John Smith","city":"Ho Chi Minh City","country":"Vietnam","match_type":"exact"},{"id":335,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},{"id":377,"display_name":"John Smith","city":"Singapore","country":"Singapore","match_type":"exact"},

# Evaluation code

In the test phase, you will be given 5 CV files with fixed names:

    CV_1.pdf, CV_2.pdf, CV_3.pdf, CV_4.pdf, CV_5.pdf

Your system must process these CVs and output a list of 5 scores,
one score per CV, in the same order:

    scores = [s1, s2, s3, s4, s5]

Each score must be a float in the range [0, 1], representing the
reliability or confidence that the CV is valid (or meets the task criteria).

The ground-truth labels are binary:

    groundtruth = [0 or 1, ..., 0 or 1]

Each CV is evaluated independently using a threshold of 0.5:

- If score > 0.5 and groundtruth == 1 → Full credit
- If score ≤ 0.5 and groundtruth == 0 → Full credit
- Otherwise → No credit

In other words, 0.5 is the decision threshold.

- Each CV contributes equally.
- Final score = (number of correct decisions) / 5


In [ ]:
# =====================================================
#  Evaluation code
# =====================================================

def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


In [ ]:
scores = ... # Your code should generate this list [0.2, 0.3, 0.4, 0.5, 0.6]
groundtruth = [1, 1, 1, 0, 0] # Do not modify

result = evaluate(scores, groundtruth)
print(result)


{'decisions': [1, 0, 1, 0, 1], 'correct': 3, 'total': 5, 'final_score': 0.6}
